# **Prototype 3D-to-2D Craniofacial Shape Analysis for Forensic Identification**

In [ ]:
!pip install nibabel SimpleITK opencv-python

In [ ]:
import nibabel as nib
import numpy as np
from google.colab import drive
import SimpleITK as sitk
import os
import pandas as pd
import cv2
import time
import re

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

## Repeatability check: repeated processing of the same scan

In [ ]:
def score_metrics(mask1, mask2):
  def dice(mask1, mask2):
    vol1 = np.sum(mask1)
    vol2 = np.sum(mask2)

    if vol1 == 0 and vol2 == 0:
        return 1.0
    if vol1 == 0 or vol2 == 0:
        return 0.0

    inter = np.sum(mask1 & mask2)
    return 2 * inter / (vol1 + vol2)

  def iou(mask1, mask2):
    inter = np.sum(mask1 & mask2)
    union = np.sum(mask1 | mask2)

    if union == 0:
        return 1.0

    return inter / union



  dice_score = dice(mask1, mask2)
  iou_score = iou(mask1, mask2)

  return dice_score, iou_score

In [ ]:
def simple_comparison_metrics(seg1_path, seg2_path):
  def load_mask(path):
    data = nib.load(path).get_fdata()
    return (data > 0).astype(np.uint8)



  def load_png_mask(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    mask = (img > 0).astype(np.uint8)
    return mask
  if seg1_path.endswith('.nii.gz') and seg2_path.endswith('.nii.gz'):
    mask1 = load_mask(seg1_path)
    mask2 = load_mask(seg2_path)
  elif seg1_path.endswith('.png') and seg2_path.endswith('.png'):
    mask1 = load_png_mask(seg1_path)
    mask2 = load_png_mask(seg2_path)
  else:
    raise ValueError(f"Unsupported file types for comparison: {seg1_path}, {seg2_path}")

  dice_score, iou_score = score_metrics(mask1, mask2)

  return dice_score, iou_score

In [ ]:
def volumes_consitency_test(first_run_path, second_run_path):
  results_list = []
  for person in os.listdir(first_run_path):
    for volume in os.listdir(f"{first_run_path}/{person}"):
      start_time = time.time()
      first_path = f"{first_run_path}/{person}/{volume}"
      second_path = f"{second_run_path}/{person}/{volume}"
      try:
        dice_score, iou_score = simple_comparison_metrics(first_path, second_path)
        end_time = time.time()
        execution_time = end_time - start_time
        # Append the dictionary to the list
        results_list.append({'person': person, 'volume': volume, 'dice_score': dice_score, 'iou_score': iou_score, 'time': execution_time})
      except ValueError as e:
        print(f"Skipping file due to unsupported type: {e}")

  # Create DataFrame from the list of dictionaries after the loop
  df = pd.DataFrame(results_list)
  return df

In [ ]:
def projections_consitency_test(first_run_path, second_run_path):
  results_list = []
  for person in os.listdir(first_run_path):
    person_path = os.path.join(first_run_path, person)
    if not os.path.isdir(person_path):
      continue
    for volume in os.listdir(person_path):
      volume_path = os.path.join(person_path, volume)
      if not os.path.isdir(volume_path):
        continue
      for projection in os.listdir(volume_path):
        start_time = time.time()
        first_path = f"{first_run_path}/{person}/{volume}/{projection}"
        second_path = f"{second_run_path}/{person}/{volume}/{projection}"
        try:
          dice_score, iou_score = simple_comparison_metrics(first_path, second_path)
          end_time = time.time()
          execution_time = end_time - start_time
          # Append the dictionary to the list
          results_list.append({'person': person, 'volume': volume, 'projection': projection, 'dice_score': dice_score, 'iou_score': iou_score, 'time': execution_time})
        except ValueError as e:
          print(f"Skipping file due to unsupported type: {e}")
  # Create DataFrame from the list of dictionaries after the loop
  df = pd.DataFrame(results_list)
  return df

In [ ]:
#Volumes consistency test
first_vol = '/content/drive/MyDrive/Pipeline/first_run/Volumes'
second_vol = '/content/drive/MyDrive/Pipeline/second_run/Volumes'
volume_comparison_result = volumes_consitency_test(first_vol, second_vol)

In [ ]:
#Projections consistency test
first_pro = '/content/drive/MyDrive/Pipeline/first_run/Projections'
second_pro = '/content/drive/MyDrive/Pipeline/second_run/Projections'
pro_comparison_result = projections_consitency_test(first_pro, second_pro)

In [ ]:
def get_folder_size(folder_path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(folder_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.isfile(fp):
                total_size += os.path.getsize(fp)
    return total_size
def get_file_size(path):
    size_bytes = os.path.getsize(path)
    return size_bytes

def format_size(bytes_size):
    for unit in ['B','KB','MB','GB','TB']:
        if bytes_size < 1024:
            return f"{bytes_size:.2f} {unit}"
        bytes_size /= 1024


In [ ]:
# Get the size
ct_p1_size = get_file_size('/content/drive/MyDrive/Pipeline/Data/Person1.nii.gz')
ct_p2_size = get_file_size('/content/drive/MyDrive/Pipeline/Data/Person2.nii.gz')
vol_p1_1_size = get_folder_size('/content/drive/MyDrive/Pipeline/first_run/Volumes/Person1')
vol_p1_2_size = get_folder_size('/content/drive/MyDrive/Pipeline/second_run/Volumes/Person1')
pro_p1_1_size = get_folder_size('/content/drive/MyDrive/Pipeline/first_run/Projections/Person1')
pro_p1_2_size = get_folder_size('/content/drive/MyDrive/Pipeline/second_run/Projections/Person1')
vol_p2_1_size = get_folder_size('/content/drive/MyDrive/Pipeline/first_run/Volumes/Person2')
vol_p2_2_size = get_folder_size('/content/drive/MyDrive/Pipeline/second_run/Volumes/Person2')
pro_p2_1_size = get_folder_size('/content/drive/MyDrive/Pipeline/first_run/Projections/Person2')
pro_p2_2_size = get_folder_size('/content/drive/MyDrive/Pipeline/second_run/Projections/Person2')


In [ ]:
#Time log path
vol_p1_1_seg_time_log_file = '/content/drive/MyDrive/Pipeline/first_run/Volumes/Person1/segmentation_log.txt'
vol_p1_2_seg_time_log_file = '/content/drive/MyDrive/Pipeline/second_run/Volumes/Person1/segmentation_log.txt'
vol_p2_1_seg_time_log_file = '/content/drive/MyDrive/Pipeline/first_run/Volumes/Person2/segmentation_log.txt'
vol_p2_2_seg_time_log_file = '/content/drive/MyDrive/Pipeline/second_run/Volumes/Person2/segmentation_log.txt'
pro_p1_1_pro_time_log_file = '/content/drive/MyDrive/Pipeline/first_run/Projections/Person1/projection_log.txt'
pro_p1_2_pro_time_log_file = '/content/drive/MyDrive/Pipeline/second_run/Projections/Person1/projection_log.txt'
pro_p2_1_pro_time_log_file = '/content/drive/MyDrive/Pipeline/first_run/Projections/Person2/projection_log.txt'
pro_p2_2_pro_time_log_file = '/content/drive/MyDrive/Pipeline/second_run/Projections/Person2/projection_log.txt'

In [ ]:
def extract_elapsed_time(log_file_path):
    if not os.path.exists(log_file_path):
        print(f"Warning: Log file not found at {log_file_path}. Returning 0.0.")
        return 0.0

    with open(log_file_path, 'r') as f:
        content = f.read()
        match = re.search(r"Elapsed Time: (\d+\.\d+) seconds", content)
        if match:
            return float(match.group(1))
        else:
            print(f"Warning: 'Elapsed Time' not found in {log_file_path}. Returning 0.0.")
            return 0.0

In [ ]:
#Extract executed time
segmention_time1_p1 = extract_elapsed_time(vol_p1_1_seg_time_log_file)
segmention_time2_p1 = extract_elapsed_time(vol_p1_2_seg_time_log_file)
segmention_time1_p2 = extract_elapsed_time(vol_p2_1_seg_time_log_file)
segmention_time2_p2 = extract_elapsed_time(vol_p2_2_seg_time_log_file)

projection_time1_p1 = extract_elapsed_time(pro_p1_1_pro_time_log_file)
projection_time2_p1 = extract_elapsed_time(pro_p1_2_pro_time_log_file)
projection_time1_p2 = extract_elapsed_time(pro_p2_1_pro_time_log_file)
projection_time2_p2 = extract_elapsed_time(pro_p2_2_pro_time_log_file)

In [ ]:

# Get total comparison times for volumes per person
vol_comp_time_series = volume_comparison_result.groupby('person')['time'].sum()

# Get total comparison times for projections per person
pro_comp_time_series = pro_comparison_result.groupby('person')['time'].sum()

# Get average dice and iou scores for volumes per person
vol_avg_metrics = volume_comparison_result.groupby('person')[['dice_score', 'iou_score']].mean()

# Get average dice and iou scores for projections per person
pro_avg_metrics = pro_comparison_result.groupby('person')[['dice_score', 'iou_score']].mean()

# Prepare data for each person
data = []
people = ['Person1', 'Person2']

for person_name in people:
    # CT size
    current_ct_size = ct_p1_size if person_name == 'Person1' else ct_p2_size

    # Volume sizes for run 1 and run 2
    current_vol_1_size = vol_p1_1_size if person_name == 'Person1' else vol_p2_1_size
    current_vol_2_size = vol_p1_2_size if person_name == 'Person1' else vol_p2_2_size

    # Projection sizes for run 1 and run 2
    current_pro_1_size = pro_p1_1_size if person_name == 'Person1' else pro_p2_1_size
    current_pro_2_size = pro_p1_2_size if person_name == 'Person1' else pro_p2_2_size

    # Segmentation times for run 1 and run 2
    seg_time1 = segmention_time1_p1 if person_name == 'Person1' else segmention_time1_p2
    seg_time2 = segmention_time2_p1 if person_name == 'Person1' else segmention_time2_p2

    # Projection times for run 1 and run 2
    proj_time1 = projection_time1_p1 if person_name == 'Person1' else projection_time1_p2
    proj_time2 = projection_time2_p1 if person_name == 'Person1' else projection_time2_p2

    # Average Dice and IoU (overall average of volume and projection comparisons)
    vol_dice = vol_avg_metrics.loc[person_name]['dice_score']
    vol_iou = vol_avg_metrics.loc[person_name]['iou_score']

    pro_dice = pro_avg_metrics.loc[person_name]['dice_score']
    pro_iou = pro_avg_metrics.loc[person_name]['iou_score']

    # Total comparison times
    total_volume_comparison_time = vol_comp_time_series.loc[person_name]
    total_pro_comparison_time = pro_comp_time_series.loc[person_name]

    data.append({
        'person': person_name,
        'ct_size': format_size(current_ct_size),
        'segmention_time1 (s)': seg_time1,
        'total_volume_size1': format_size(current_vol_1_size),
        'projection_time1 (s)': proj_time1,
        'total_projection_size1': format_size(current_pro_1_size),
        'segmention_time2 (s)': seg_time2,
        'total_volume_size2': format_size(current_vol_2_size),
        'projection_time2 (s)': proj_time2,
        'total_projection_size2': format_size(current_pro_2_size),
        'vol_compare_dice': vol_dice,
        'vol_compare_iou': vol_iou,
        'pro_compare_dice': pro_dice,
        'pro_compare_iou': pro_iou,
        'total_volume_comparison_time (s)': total_volume_comparison_time,
        'total_pro_comparison_time (s)': total_pro_comparison_time
    })

# Create DataFrame
summary_df = pd.DataFrame(data)

# Display the DataFrame
display(summary_df)

In [ ]:
#Export the summary_df


In [ ]:
output_path = '/content/drive/MyDrive/Pipeline/summary_report.csv'
summary_df.to_csv(output_path, index=False)
print(f"DataFrame đã được xuất thành công tới: {output_path}")

## Exploratory between-case comparison: Person1 vs Person2

In [ ]:
def imposters_comparison_metrics(seg1_path, seg2_path):
    def load_mask(path):
        img = sitk.ReadImage(path)
        return img

    def resample_to_reference(moving_img, reference_img):
        return sitk.Resample(
            moving_img,
            reference_img,
            sitk.Transform(),
            sitk.sitkNearestNeighbor,
            0,
            moving_img.GetPixelID()
        )

    def nii_to_binary_mask(img):
        arr = sitk.GetArrayFromImage(img)
        return (arr > 0).astype(np.uint8)

    def load_png_mask(path):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Cannot read PNG file: {path}")
        mask = (img > 0).astype(np.uint8)
        return mask

    if seg1_path.endswith('.nii.gz') and seg2_path.endswith('.nii.gz'):
        seg1_img = load_mask(seg1_path)
        seg2_img = load_mask(seg2_path)

        seg2_img = resample_to_reference(seg2_img, seg1_img)

        mask1 = nii_to_binary_mask(seg1_img)
        mask2 = nii_to_binary_mask(seg2_img)

    elif seg1_path.endswith('.png') and seg2_path.endswith('.png'):
        mask1 = load_png_mask(seg1_path)
        mask2 = load_png_mask(seg2_path)

        if mask1.shape != mask2.shape:
            # Resize mask2 to match mask1's shape
            mask2 = cv2.resize(mask2, (mask1.shape[1], mask1.shape[0]), interpolation=cv2.INTER_NEAREST)

    else:
        raise ValueError(f"Unsupported file types for comparison: {seg1_path}, {seg2_path}")

    dice_score, iou_score = score_metrics(mask1, mask2)

    return dice_score, iou_score

In [ ]:
def volumes_differ_test(p1_path, p2_path):
  results_list = []
  for volume_file_name in os.listdir(p1_path):
    # Only process NII.GZ files
    if not volume_file_name.endswith('.nii.gz'):
        continue

    start_time = time.time()
    first_file_path = os.path.join(p1_path, volume_file_name)
    second_file_path = os.path.join(p2_path, volume_file_name)
    try:
      dice_score, iou_score = imposters_comparison_metrics(first_file_path, second_file_path)
      end_time = time.time()
      execution_time = end_time - start_time
      results_list.append({'volume': volume_file_name, 'dice_score': dice_score, 'iou_score': iou_score, 'time': execution_time})
    except ValueError as e:
      print(f"Skipping file due to unsupported type: {e}")

  df = pd.DataFrame(results_list)
  return df

In [ ]:
def projections_differ_test(p1_person_pro_path, p2_person_pro_path):
  results_list = []
  person1_name = os.path.basename(p1_person_pro_path)
  person2_name = os.path.basename(p2_person_pro_path)

  for volume_name in os.listdir(p1_person_pro_path):
    volume1_dir = os.path.join(p1_person_pro_path, volume_name)
    volume2_dir = os.path.join(p2_person_pro_path, volume_name)

    if not os.path.isdir(volume1_dir) or not os.path.isdir(volume2_dir):
        continue

    for projection_file_name in os.listdir(volume1_dir):
      # Only process PNG files
      if not projection_file_name.endswith('.png'):
        continue

      start_time = time.time()
      file1_path = os.path.join(volume1_dir, projection_file_name)
      file2_path = os.path.join(volume2_dir, projection_file_name)

      try:
        dice_score, iou_score = imposters_comparison_metrics(file1_path, file2_path)
        end_time = time.time()
        execution_time = end_time - start_time
        results_list.append({
            'person1': person1_name,
            'person2': person2_name,
            'volume': volume_name,
            'projection': projection_file_name,
            'dice_score': dice_score,
            'iou_score': iou_score,
            'time': execution_time
        })
      except ValueError as e:
        print(f"Skipping file due to unsupported type: {e}")
  df = pd.DataFrame(results_list)
  return df

In [ ]:
p1_vol = '/content/drive/MyDrive/Pipeline/first_run/Volumes/Person1'
p2_vol = '/content/drive/MyDrive/Pipeline/first_run/Volumes/Person2'
volume_difer_result = volumes_differ_test(p1_vol,p2_vol)
display(volume_difer_result)

In [ ]:
volume_difer_result[['dice_score','iou_score']].mean()

In [ ]:
volume_difer_result['time'].sum()

In [ ]:
p1_pro_path = '/content/drive/MyDrive/Pipeline/first_run/Projections/Person1'
p2_pro_path = '/content/drive/MyDrive/Pipeline/first_run/Projections/Person2'
pro_differ_result = projections_differ_test(p1_pro_path, p2_pro_path)
display(pro_differ_result)

In [ ]:
pro_differ_result[['dice_score','iou_score']].mean()

In [ ]:
pro_differ_result['time'].sum()

In [ ]:
# Get average dice and iou scores for volumes between different persons
vol_differ_avg_metrics = volume_difer_result[['dice_score', 'iou_score']].mean()

# Get average dice and iou scores for projections between different persons
pro_differ_avg_metrics = pro_differ_result[['dice_score', 'iou_score']].mean()

combined_p1vsp2_results = pd.DataFrame({
    'comparison_type': ['Volumes (Person1 vs Person2)', 'Projections (Person1 vs Person2)'],
    'avg_dice_score': [vol_differ_avg_metrics['dice_score'], pro_differ_avg_metrics['dice_score']],
    'avg_iou_score': [vol_differ_avg_metrics['iou_score'], pro_differ_avg_metrics['iou_score']],
    'total_comparison_time_seconds': [volume_difer_result['time'].sum(), pro_differ_result['time'].sum()]
})

# Display the combined DataFrame
display(combined_p1vsp2_results)

In [ ]:
output_path_p1vsp2 = '/content/drive/MyDrive/Pipeline/p1vsp2_result.csv'
combined_p1vsp2_results.to_csv(output_path_p1vsp2, index=False)
print(f"DataFrame for Person1 vs Person2 comparison exported to: {output_path_p1vsp2}")